In [1]:
import pandas as pd
import numpy as np

In [4]:
data = pd.read_csv("Datasets/DataCoSupplyChainDataset.csv", encoding="latin1")

data.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [5]:
# 1. Drop redundant or empty columns (and Order Zipcode for compliance)
cols_to_drop = [
    'Customer Email', 
    'Customer Password', 
    'Product Image', 
    'Product Description', 
    'Order Zipcode'
]
# We use 'data' instead of 'df' to match your previous cell
data_cleaned = data.drop(columns=[col for col in cols_to_drop if col in data.columns])

# 2. Drop the handful of rows missing Zipcode and Last Name
data_cleaned = data_cleaned.dropna(subset=['Customer Zipcode', 'Customer Lname'])

# 3. Convert text strings to standard datetime objects for time-series analysis
data_cleaned['order_date'] = pd.to_datetime(data_cleaned['order date (DateOrders)'])
data_cleaned['shipping_date'] = pd.to_datetime(data_cleaned['shipping date (DateOrders)'])

# 4. Feature Engineering: Calculate delays and processing time
data_cleaned['shipping_delay_days'] = data_cleaned['Days for shipping (real)'] - data_cleaned['Days for shipment (scheduled)']
data_cleaned['processing_time_hours'] = (data_cleaned['shipping_date'] - data_cleaned['order_date']).dt.total_seconds() / 3600.0

# 5. Round all major financial columns to two decimal places
numeric_financial_cols = [
    'Benefit per order', 'Sales per customer', 'Sales', 
    'Order Item Total', 'Order Profit Per Order', 'Product Price'
]
for col in numeric_financial_cols:
    if col in data_cleaned.columns:
        data_cleaned[col] = data_cleaned[col].round(2)

# Output the results to verify the data was cleaned
print(f"Original shape: {data.shape}")
print(f"Cleaned shape: {data_cleaned.shape}")

# Preview the newly created columns
data_cleaned[['order_date', 'shipping_date', 'shipping_delay_days', 'processing_time_hours']].head()

Original shape: (180519, 53)
Cleaned shape: (180508, 52)


,order_date,shipping_date,shipping_delay_days,processing_time_hours
0,2018-01-31 22:56:00,2018-02-03 22:56:00,-1,72.0
1,2018-01-13 12:27:00,2018-01-18 12:27:00,1,120.0
2,2018-01-13 12:06:00,2018-01-17 12:06:00,0,96.0
3,2018-01-13 11:45:00,2018-01-16 11:45:00,-1,72.0
4,2018-01-13 11:24:00,2018-01-15 11:24:00,-2,48.0


In [6]:
# Export the cleaned data to a new CSV file
data_cleaned.to_csv("Datasets/Cleaned_DataCo_SupplyChain.csv", index=False)

print("Cleaned dataset successfully exported!")

Cleaned dataset successfully exported!


In [7]:
import sqlite3
import pandas as pd

# 1. Create a local, lightweight SQL database right in your project folder
conn = sqlite3.connect('Datasets/supply_chain.db')

# 2. Push your cleaned dataframe into this new SQL database
data_cleaned.to_sql('Cleaned_DataCo_SupplyChain', conn, if_exists='replace', index=False)

180508

In [ ]:
# 3. Paste your SQL query inside a Python string
query = """
SELECT 
    "Category Name" AS product_category,
    "Shipping Mode" AS shipping_mode,
    COUNT("Order Id") AS total_orders,
    ROUND(AVG(shipping_delay_days), 2) AS avg_delay_in_days,
    ROUND(AVG("Late_delivery_risk") * 100, 2) AS late_delivery_risk_percentage
FROM 
    Cleaned_DataCo_SupplyChain
GROUP BY 
    "Category Name",
    "Shipping Mode"
HAVING 
    AVG(shipping_delay_days) > 0
ORDER BY 
    late_delivery_risk_percentage DESC;
"""

# 4. Execute the SQL query and display the results!
sql_results = pd.read_sql(query, conn)
sql_results.head(10)

,product_category,shipping_mode,total_orders,avg_delay_in_days,late_delivery_risk_percentage
0,Basketball,Second Class,10,3.1,100.00
1,Books,First Class,53,1.0,100.00
2,Garden,First Class,73,1.0,100.00
3,Golf Bags & Carts,First Class,19,1.0,100.00
4,Golf Bags & Carts,Same Day,1,1.0,100.00
5,Lacrosse,First Class,51,1.0,100.00
6,Soccer,First Class,15,1.0,100.00
7,Strength Training,First Class,14,1.0,100.00
8,Golf Shoes,First Class,78,1.0,98.72
9,Golf Apparel,First Class,58,1.0,98.28


In [ ]:

    #adding comment to push file again